# DIMER Language-Model Fine-Tuning — Standalone Colab

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/language-model-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/language-model-pipeline/blob/main/tutorials/language_model_finetuning_colab.ipynb)

**E2E:** approved pinned base → sample/BYOD → validation → baseline generation → QLoRA SFT → before/after + new prompts → adapter-first export → fresh reload.

Loss/perplexity are optimization evidence, not task-quality scores. Prompt comparisons are illustrative. The PEFT bundle contains no base weights or training rows.

**AI provenance:** OpenAI / ChatGPT — GPT-5.6 Sol High, Builder role; not independent reviewer sign-off.

## 1. Runtime

In [ ]:
%pip -q install transformers==4.57.1 tokenizers==0.22.1 huggingface-hub==0.36.0 peft==0.18.0 accelerate==1.11.0 bitsandbytes==0.49.0 safetensors==0.8.0 "datasets>=3,<5"


In [ ]:
import gc,hashlib,json,math,os,platform,random,shutil,stat,time,zipfile
from pathlib import Path
import pandas as pd,torch
from datasets import load_dataset
from huggingface_hub import HfApi
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training,PeftModel
if not torch.cuda.is_available(): raise RuntimeError("Use a Colab GPU runtime")
GPU_NAME=torch.cuda.get_device_name(0); GPU_VRAM_GB=torch.cuda.get_device_properties(0).total_memory/1024**3
AI_PROVENANCE={"provider":"OpenAI","product":"ChatGPT","model":"GPT-5.6 Sol High","role":"Builder","note":"Not independent reviewer sign-off"}
print(platform.python_version(),torch.__version__,GPU_NAME,f"{GPU_VRAM_GB:.2f} GiB")


## 2. Approved base model

No arbitrary Hugging Face IDs. Revisions are immutable and `trust_remote_code=False` is fixed. Current user-facing measured profiles support QLoRA; unmeasured LoRA fails closed.

In [ ]:
TUTORIAL_REGISTRY={
"qwen3-1.7b":("Qwen/Qwen3-1.7B","70d244cc86ccca08cf5af4e1e306ecf908b1ad5e","apache-2.0",9.3),
"qwen3-4b":("Qwen/Qwen3-4B","1cfa9a7208912126459214e8b04321603b3df60c","apache-2.0",11.5),
"granite-4.1-3b":("ibm-granite/granite-4.1-3b","c0650403e44e78ec0262dab1c90914c65b196c4e","apache-2.0",8.7)}
BASE_MODEL_KEY="qwen3-1.7b" # @param ["qwen3-1.7b","qwen3-4b","granite-4.1-3b"]
TRAINING_METHOD="qlora"; MAX_SEQUENCE_LENGTH=512 # @param {type:"integer"}
EPOCHS=1; LEARNING_RATE=0.0002; LORA_RANK=8; SEED=42
model_id,revision,base_license,min_vram=TUTORIAL_REGISTRY[BASE_MODEL_KEY]
if TRAINING_METHOD!="qlora": raise RuntimeError("LoRA profile unmeasured; use QLoRA")
if GPU_VRAM_GB<min_vram: raise RuntimeError(f"Need registry minimum {min_vram} GiB")
print(BASE_MODEL_KEY,model_id,revision,"preflight PASS")


## 3. Data
Default: `jpaulpoliquit/ph-sft-ai-authored-v1` (Filipino/English, Apache-2.0, AI-authored tutorial seed; **not a benchmark**). Dolly is the English fallback. BYOD accepts JSONL or one safe ZIP with `train.jsonl` plus optional validation/test. Supported records: `messages`, `prompt/completion`, `instruction/input/output`.

In [ ]:
DATA_SOURCE="Sample: Filipino SFT" # @param ["Sample: Filipino SFT","Sample: Dolly","Bring Your Own Dataset"]
SAMPLE_LIMIT=120; MAX_TOTAL_TRAIN_TOKENS = 50_000_000; W=Path("/content/lm-sft"); shutil.rmtree(W,ignore_errors=True); W.mkdir()
def canonical(r):
 if "messages" in r: m=[{"role":x["role"],"content":str(x["content"])} for x in r["messages"]]
 elif "prompt" in r and ("completion" in r or "response" in r): m=[{"role":"user","content":str(r["prompt"])},{"role":"assistant","content":str(r.get("completion",r.get("response")))}]
 elif "instruction" in r and ("output" in r or "response" in r):
  q=str(r["instruction"])+(f"\n\n{r.get('input') or r.get('context')}" if r.get("input") or r.get("context") else ""); m=[{"role":"user","content":q},{"role":"assistant","content":str(r.get("output",r.get("response")))}]
 else: raise ValueError("Unsupported SFT schema")
 if any(x["role"] not in {"system","user","assistant"} for x in m) or not any(x["role"]=="assistant" and x["content"].strip() for x in m): raise ValueError("Invalid roles/assistant target")
 return {"messages":m}
def fp(r): return hashlib.sha256(json.dumps(r,sort_keys=True,ensure_ascii=False,separators=(",",":")).encode()).hexdigest()
def safe_extract_zip(p,d):
 d=d.resolve(); d.mkdir(exist_ok=True);
 with zipfile.ZipFile(p) as z:
  total=0
  for i in z.infolist():
   q=Path(i.filename); total+=i.file_size
   if q.is_absolute() or ".." in q.parts or stat.S_ISLNK((i.external_attr>>16)&0xffff) or total>1024**3: raise ValueError("Unsafe ZIP")
   t=(d/q).resolve()
   if t!=d and d not in t.parents: raise ValueError("ZIP escape")
  z.extractall(d)
 return d
if DATA_SOURCE.startswith("Sample"):
 if DATA_SOURCE=="Sample: Filipino SFT": dsid="jpaulpoliquit/ph-sft-ai-authored-v1"; rev=HfApi().dataset_info(dsid).sha; lic="apache-2.0"
 else: dsid="databricks/databricks-dolly-15k"; rev="bdd27f4d94b9c1f951818a7da7fd7aeea5dbff1a"; lic="cc-by-sa-3.0"
 rows=sorted([canonical(dict(x)) for x in load_dataset(dsid,revision=rev,split="train")],key=fp)[:SAMPLE_LIMIT]; cut=max(1,len(rows)//5); SPLITS={"train":rows[cut:],"validation":rows[:cut]}; DATASET_PROVENANCE={"source":dsid,"revision":rev,"license":lic,"usage":"tutorial-training-not-benchmark"}
else:
 from google.colab import files
 u=files.upload(); names=list(u); root=W/"byod"; root.mkdir()
 if len(names)==1 and names[0].endswith(".zip"): z=W/"data.zip"; z.write_bytes(u[names[0]]); root=safe_extract_zip(z,root)
 else:
  for n,b in u.items(): (root/Path(n).name).write_bytes(b)
 def read(p): return [canonical(json.loads(x)) for x in p.read_text().splitlines() if x.strip()]
 if not (root/"train.jsonl").exists(): raise ValueError("BYOD requires train.jsonl")
 if (root/"validation.jsonl").exists() and (root/"val.jsonl").exists(): raise ValueError("Ambiguous validation split")
 SPLITS={"train":read(root/"train.jsonl")}; v=root/("validation.jsonl" if (root/"validation.jsonl").exists() else "val.jsonl")
 if v.exists(): SPLITS["validation"]=read(v)
 if (root/"test.jsonl").exists(): SPLITS["test"]=read(root/"test.jsonl")
 DATASET_PROVENANCE={"source":"BYOD","usage":"user-provided"}
for k,v in SPLITS.items():
 if len(v)!=len({fp(x) for x in v}): print(f"Warning: exact duplicates in {k}; none removed")
for a,b in [("train","validation"),("train","test"),("validation","test")]:
 if a in SPLITS and b in SPLITS and {fp(x) for x in SPLITS[a]}&{fp(x) for x in SPLITS[b]}: raise ValueError(f"Split leakage {a}/{b}")
if "validation" not in SPLITS: rows=SPLITS["train"]; cut=max(1,len(rows)//5); SPLITS={**SPLITS,"train":rows[cut:],"validation":rows[:cut]}
DATASET_DIGEST=hashlib.sha256("".join(fp(x) for k in sorted(SPLITS) for x in SPLITS[k]).encode()).hexdigest(); print({k:len(v) for k,v in SPLITS.items()},DATASET_DIGEST[:16])


## 4. Tokenizer validation + assistant-only masking
Tokenizer checks run before weight download. Overlength samples fail; no silent truncation. Only assistant spans contribute loss.

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(model_id,revision=revision,trust_remote_code=False)
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
if not tokenizer.chat_template: raise ValueError("No chat template")
IGNORE=-100
def enc(s): return tokenizer(s,add_special_tokens=False)["input_ids"]
def rend(m,g=False): return tokenizer.apply_chat_template(m,tokenize=False,add_generation_prompt=g)
def build_masked_example(r):
 m=r["messages"]; ids=enc(rend(m)); labels=[IGNORE]*len(ids)
 if len(ids)>MAX_SEQUENCE_LENGTH: raise ValueError("DATASET_SEQUENCE_TOO_LONG")
 for i,x in enumerate(m):
  if x["role"]=="assistant":
   a,b=enc(rend(m[:i],True)),enc(rend(m[:i+1]));
   if ids[:len(a)]!=a or ids[:len(b)]!=b: raise ValueError("Chat template not prefix-stable")
   labels[len(a):len(b)]=ids[len(a):len(b)]
 if all(x==IGNORE for x in labels): raise ValueError("No supervised tokens")
 return ids,labels
MASKED={k:[build_masked_example(r) for r in v] for k,v in SPLITS.items()}; totals={k:sum(len(x[0]) for x in v) for k,v in MASKED.items()}
if totals["train"]>MAX_TOTAL_TRAIN_TOKENS: raise ValueError("DATASET_TOKEN_BUDGET_EXCEEDED")
print(totals)


## 5. Baseline → QLoRA fine-tuning → comparison

In [ ]:
bf16=torch.cuda.is_bf16_supported(); q=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.bfloat16 if bf16 else torch.float16)
base_model=AutoModelForCausalLM.from_pretrained(model_id,revision=revision,trust_remote_code=False,dtype=torch.bfloat16 if bf16 else torch.float16,quantization_config=q,device_map={"":0})
if getattr(base_model.config,"_commit_hash",revision)!=revision: raise RuntimeError("revision mismatch")
def generate(m,p,n=96):
 s=rend([{"role":"user","content":p}],True); x=tokenizer(s,return_tensors="pt",add_special_tokens=False).to("cuda")
 with torch.no_grad(): y=m.generate(**x,max_new_tokens=n,do_sample=False,pad_token_id=tokenizer.pad_token_id)
 return tokenizer.decode(y[0,x["input_ids"].shape[1]:],skip_special_tokens=True).strip()
PROMPTS=["Ipaliwanag sa simpleng Filipino kung ano ang machine learning.","Magbigay ng tatlong paraan para mabawasan ang basura sa opisina."]
BASELINE_OUTPUTS=[generate(base_model,p) for p in PROMPTS]
targets=sorted({n.rsplit(".",1)[-1] for n,_ in base_model.named_modules()}&{"q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"})
if not targets: raise RuntimeError("No LoRA target modules found")
model=get_peft_model(prepare_model_for_kbit_training(base_model),LoraConfig(r=LORA_RANK,lora_alpha=16,lora_dropout=.05,bias="none",task_type="CAUSAL_LM",target_modules=targets))
def batch(x):
 ids,lab=x; return {"input_ids":torch.tensor([ids],device="cuda"),"labels":torch.tensor([lab],device="cuda"),"attention_mask":torch.ones((1,len(ids)),dtype=torch.long,device="cuda")}
def evaluate(xs):
 model.eval(); loss=toks=0
 with torch.no_grad():
  for x in xs:
   b=batch(x); o=model(**b); n=int((b["labels"]!=IGNORE).sum()); loss+=float(o.loss)*n; toks+=n
 model.train(); return loss/toks if toks else None
opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=LEARNING_RATE); started=time.time(); torch.cuda.reset_peak_memory_stats(); random.seed(SEED)
for e in range(EPOCHS):
 xs=MASKED["train"][:]; random.shuffle(xs); loss=toks=0; opt.zero_grad()
 for i,x in enumerate(xs,1):
  b=batch(x); o=model(**b); (o.loss/2).backward(); n=int((b["labels"]!=IGNORE).sum()); loss+=float(o.loss)*n; toks+=n
  if i%2==0 or i==len(xs): opt.step(); opt.zero_grad()
 val=evaluate(MASKED["validation"]); print(e+1,loss/toks,val)
METRICS={"trainLoss":loss/toks,"validationLoss":val,"testLoss":evaluate(MASKED.get("test",[])),"validationPerplexity":math.exp(val) if val<20 else None,"wallSeconds":time.time()-started,"peakGpuMemoryBytes":torch.cuda.max_memory_allocated()}
ADAPTED_OUTPUTS=[generate(model,p) for p in PROMPTS]; display(pd.DataFrame({"prompt":PROMPTS,"base":BASELINE_OUTPUTS,"adapted":ADAPTED_OUTPUTS})); print(METRICS)


In [ ]:
RUN_NEW_PROMPT_INFERENCE=True # @param {type:"boolean"}
NEW_PROMPTS=["Sumulat ng maikling payo para sa isang estudyanteng nagsisimula sa AI."]
if RUN_NEW_PROMPT_INFERENCE: display(pd.DataFrame({"prompt":NEW_PROMPTS,"response":[generate(model,p) for p in NEW_PROMPTS]}))


## 6. Adapter-first export + fresh reload
The manifest hashes every published file. Provenance records the exact base revision, dataset digest and AI provenance. The training data itself is excluded.

In [ ]:
S=Path("/content/dimer-lm-adapter.staging"); A=Path("/content/dimer-lm-adapter"); Z=Path("/content/dimer-language-model-adapter.zip"); shutil.rmtree(S,ignore_errors=True); shutil.rmtree(A,ignore_errors=True); S.mkdir()
model.save_pretrained(S,safe_serialization=True); tokenizer.save_pretrained(S/"tokenizer")
PROVENANCE={"artifactFormat":"peft_adapter","artifactFormatVersion":1,"baseModel":model_id,"baseModelRevision":revision,"baseModelRevisionExpected":revision,"baseModelLicense":base_license,"modelKey":BASE_MODEL_KEY,"trustRemoteCode":False,"datasetDigest":DATASET_DIGEST,"dataset":DATASET_PROVENANCE,"training":{"method":TRAINING_METHOD,"epochs":EPOCHS},"aiProvenance":AI_PROVENANCE}
(S/"metrics.json").write_text(json.dumps(METRICS)); (S/"provenance.json").write_text(json.dumps(PROVENANCE)); (S/"MODEL_CARD.md").write_text(f"# Adapter for {model_id}\n\nBase revision: `{revision}`. Optimization metrics are not task-quality evidence.\n")
def sha(p):
 h=hashlib.sha256(); h.update(p.read_bytes()); return h.hexdigest()
records=[{"path":p.relative_to(S).as_posix(),"bytes":p.stat().st_size,"sha256":sha(p)} for p in sorted(S.rglob("*")) if p.is_file() and p.name!="artifact-manifest.json"]
(S/"artifact-manifest.json").write_text(json.dumps({"format":"peft_adapter","formatVersion":1,"files":records,"totalBytes":sum(x["bytes"] for x in records)}));
for r in records:
 if sha(S/r["path"])!=r["sha256"]: raise RuntimeError("artifact-manifest.json verification failed")
# Fresh base + adapter reload, from disk, before publication.
del model,base_model; gc.collect(); torch.cuda.empty_cache(); rt=AutoTokenizer.from_pretrained(S/"tokenizer")
rb=AutoModelForCausalLM.from_pretrained(model_id,revision=revision,trust_remote_code=False,dtype=torch.bfloat16 if bf16 else torch.float16,device_map={"":0})
if getattr(rb.config,"_commit_hash",revision)!=revision: raise RuntimeError("Fresh reload revision mismatch")
rm=PeftModel.from_pretrained(rb,S); tokenizer=rt; smoke=generate(rm,"Kumusta! Sagutin sa isang maikling pangungusap.",32)
if not smoke: raise RuntimeError("Fresh reload failed")
print("✓ Fresh base + adapter reload",smoke); os.replace(S,A)
with zipfile.ZipFile(Z,"w",zipfile.ZIP_STORED) as z:
 for p in A.rglob("*"):
  if p.is_file(): z.write(p,p.relative_to(A).as_posix())
print("Artifact SHA-256",sha(Z)); from google.colab import files; files.download(str(Z))


## Result
A successful run verifies pinned model resolution, structural/tokenizer checks, assistant-only QLoRA SFT, generation, new-prompt inference, hashed adapter export and fresh reload. It does **not** establish universal quality, safety, fairness or deployment fitness.